In [2]:
import pandas as pd
import numpy as np

In [3]:
#importing cleand data
users = pd.read_csv("../Dataset/Cleaned_Data/users_clean.csv")
cards = pd.read_csv("../Dataset/Cleaned_Data/cards_clean.csv")
transactions = pd.read_csv("../Dataset/Cleaned_Data/transactions_clean.csv")

In [4]:
# commbining transaction date (year.month,day)
transactions["Transaction Date"] = pd.to_datetime(
    transactions[["Year", "Month", "Day"]]
)

In [5]:
transactions[["Year", "Month", "Day", "Transaction Date"]].head()

,Year,Month,Day,Transaction Date
0,2002,9,1,2002-09-01
1,2002,9,1,2002-09-01
2,2002,9,2,2002-09-02
3,2002,9,2,2002-09-02
4,2002,9,3,2002-09-03


In [6]:
# extract time
#1. transaction hours
transactions["Transaction Hour"] = pd.to_datetime(
    transactions["Time"],
    format="%H:%M"
).dt.hour

In [7]:
#2. day name
transactions["Day Name"] = transactions["Transaction Date"].dt.day_name()

In [8]:
#3. month name
transactions["Month Name"] = transactions["Transaction Date"].dt.month_name()

In [9]:
#4. quater
transactions["Quarter"] = transactions["Transaction Date"].dt.quarter

In [10]:
#5. weekend
transactions["Is Weekend"] = np.where(
    transactions["Day Name"].isin(["Saturday", "Sunday"]),
    "Yes",
    "No"
)

In [11]:
# time of day
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

transactions["Time of Day"] = transactions["Transaction Hour"].apply(get_time_of_day)

In [12]:
# transaction amount category
def amount_category(amount):
    if amount < 0:
        return "Refund / Reversal"
    elif amount <= 9.20:
        return "Low"
    elif amount <= 65.06:
        return "Medium"
    elif amount <= 146.82:
        return "High"
    else:
        return "Very High"

transactions["Amount Category"] = transactions["Amount"].apply(amount_category)

In [13]:
# customer age group
def age_group(age):
    if age <= 25:
        return "18-25"
    elif age <= 35:
        return "26-35"
    elif age <= 45:
        return "36-45"
    elif age <= 60:
        return "46-60"
    else:
        return "60+"

users["Age Group"] = users["Current Age"].apply(age_group)

In [14]:
#FICO score category
def fico_category(score):
    if score >= 800:
        return "Excellent"
    elif score >= 740:
        return "Very Good"
    elif score >= 670:
        return "Good"
    elif score >= 580:
        return "Fair"
    else:
        return "Poor"

users["FICO Category"] = users["FICO Score"].apply(fico_category)

In [15]:
# debt to income ratio
users["Debt to Income Ratio"] = (
    users["Total Debt"] / users["Yearly Income - Person"]
).round(2)

In [16]:
# imcome group
def income_group(income):
    if income <= 32818.5:
        return "Low"
    elif income <= 40744.5:
        return "Middle"
    elif income <= 52698.5:
        return "Upper Middle"
    else:
        return "High"

users["Income Group"] = users["Yearly Income - Person"].apply(income_group)

In [17]:
# validating column
transactions[[
    "Transaction Date",
    "Transaction Hour",
    "Day Name",
    "Month Name",
    "Quarter",
    "Is Weekend",
    "Time of Day",
    "Amount Category"
]].head()

,Transaction Date,Transaction Hour,Day Name,Month Name,Quarter,Is Weekend,Time of Day,Amount Category
0,2002-09-01,6,Sunday,September,3,Yes,Morning,High
1,2002-09-01,6,Sunday,September,3,Yes,Morning,Medium
2,2002-09-02,6,Monday,September,3,No,Morning,High
3,2002-09-02,17,Monday,September,3,No,Evening,High
4,2002-09-03,6,Tuesday,September,3,No,Morning,High


In [18]:
users[[
    "Current Age",
    "Age Group",
    "FICO Score",
    "FICO Category",
    "Yearly Income - Person",
    "Income Group",
    "Debt to Income Ratio"
]].head()

,Current Age,Age Group,FICO Score,FICO Category,Yearly Income - Person,Income Group,Debt to Income Ratio
0,53,46-60,787,Very Good,59696.0,High,2.14
1,53,46-60,701,Good,77254.0,High,2.48
2,81,60+,698,Good,33483.0,Middle,0.01
3,63,60+,722,Good,249925.0,High,0.81
4,43,36-45,675,Good,109687.0,High,1.68


In [19]:
# by the help of this 4 metric below we update the amount and income group category
transactions["Amount"].describe()

count    2.438683e+07
mean     4.363393e+01
std      8.202244e+01
min     -5.000000e+02
25%      9.200000e+00
50%      3.014000e+01
75%      6.506000e+01
max      1.239050e+04
Name: Amount, dtype: float64

In [20]:
transactions["Amount"].quantile([0.25,0.50,0.75,0.90,0.95])

0.25      9.20
0.50     30.14
0.75     65.06
0.90    106.10
0.95    146.82
Name: Amount, dtype: float64

In [21]:
users["Yearly Income - Person"].describe()

count      2000.000000
mean      45715.882000
std       22992.615456
min           1.000000
25%       32818.500000
50%       40744.500000
75%       52698.500000
max      307018.000000
Name: Yearly Income - Person, dtype: float64

In [22]:
users["Yearly Income - Person"].quantile([0.25,0.50,0.75])

0.25    32818.5
0.50    40744.5
0.75    52698.5
Name: Yearly Income - Person, dtype: float64

In [23]:
# checking why there is negative amount appears
transactions[transactions["Amount"] < 0].head(5)

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,...,Errors?,Is Fraud?,Transaction Date,Transaction Hour,Day Name,Month Name,Quarter,Is Weekend,Time of Day,Amount Category
32,0,0,2002,9,11,13:17,-99.0,Swipe Transaction,2027553650310142703,Mira Loma,...,NaN,No,2002-09-11,13,Wednesday,September,3,No,Afternoon,Refund / Reversal
72,0,0,2002,9,25,13:14,-100.0,Swipe Transaction,-1288082279022882052,La Verne,...,NaN,No,2002-09-25,13,Wednesday,September,3,No,Afternoon,Refund / Reversal
116,0,0,2002,10,13,13:15,-99.0,Swipe Transaction,-1288082279022882052,La Verne,...,NaN,No,2002-10-13,13,Sunday,October,4,Yes,Afternoon,Refund / Reversal
122,0,0,2002,10,15,13:11,-88.0,Swipe Transaction,-4334232547381218591,Lincoln,...,NaN,No,2002-10-15,13,Tuesday,October,4,No,Afternoon,Refund / Reversal
127,0,0,2002,10,16,11:54,-207.0,Swipe Transaction,7834055923142137930,Tilden,...,NaN,No,2002-10-16,11,Wednesday,October,4,No,Morning,Refund / Reversal


In [24]:
# count of negative transaction
transactions[transactions["Amount"] < 0].shape

(1244676, 23)

In [25]:
# Fraud among negative transactions
transactions[transactions["Amount"] < 0]["Is Fraud?"].value_counts()

Is Fraud?
No     1243551
Yes       1125
Name: count, dtype: int64

In [26]:
# Errors among negative transactions
transactions[transactions["Amount"] < 0]["Errors?"].value_counts(dropna=False)

Errors?
NaN                                      1227167
Insufficient Balance                       12321
Bad PIN                                     2646
Technical Glitch                            2167
Bad Zipcode                                  286
Bad PIN,Insufficient Balance                  23
Bad CVV                                       22
Insufficient Balance,Technical Glitch         19
Bad Card Number                                9
Bad Expiration                                 7
Bad PIN,Technical Glitch                       7
Bad Zipcode,Insufficient Balance               1
Bad Card Number,Insufficient Balance           1
Name: count, dtype: int64

In [32]:
# adding user index 
users.insert(0, "User", range(len(users)))

In [33]:
users.head(5)

,User,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,...,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,Age Group,FICO Category,Debt to Income Ratio,Income Group
0,0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,...,-117.76,29278.0,59696.0,127613.0,787,5,46-60,Very Good,2.14,High
1,1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,...,-73.74,37891.0,77254.0,191349.0,701,5,46-60,Good,2.48,High
2,2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,...,-117.89,22681.0,33483.0,196.0,698,5,60+,Good,0.01,Middle
3,3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,...,-73.99,163145.0,249925.0,202328.0,722,4,60+,Good,0.81,High
4,4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,...,-122.44,53797.0,109687.0,183855.0,675,1,36-45,Good,1.68,High


In [35]:
# saving enhanced
users.to_csv(
    "../Dataset/Cleaned_Data/users_enhanced.csv",
    index=False,
    encoding="utf-8"
)

cards.to_csv(
    "../Dataset/Cleaned_Data/cards_enhanced.csv",
    index=False,
    encoding="utf-8"
)

transactions.to_csv(
    "../Dataset/Cleaned_Data/transactions_enhanced.csv",
    index=False,
    encoding="utf-8"
)